In [ ]:
#importing the required libraries which will be used throughout the implementation 

import socket
import threading
import json
import sqlite3


PORT = 5050
SERVER = socket.gethostbyname(socket.gethostname()) #used to get the host information based on the device running the code automatically 
ADDR = (SERVER, PORT)
HEADER = 64
FORMAT = 'utf-8'
DB_FILE = "cinema.db"

def init_database():  #creating the tables to store the movies and sales data respectively 
    try:
        conn = sqlite3.connect(DB_FILE)
        cursor = conn.cursor()
       
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS movies (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                title TEXT,
                cinema_room INTEGER,
                release_date TEXT,
                end_date TEXT,
                tickets_available INTEGER,
                ticket_price REAL
            )
        ''')

        cursor.execute('''
            CREATE TABLE IF NOT EXISTS sales (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                movie_id INTEGER,
                customer_name TEXT,
                number_of_tickets INTEGER,
                total REAL
            )
        ''')

        cursor.execute("SELECT COUNT(*) FROM movies")
        if cursor.fetchone()[0] == 0:
            current_movies = [
                ("Marvel Thunderbolts* (IMAX 3D Screening)", 1, "02/05/2025", "01/07/2025", 500, 150.00),
                ("Sinners (Dolby Atmos Screening)", 2, "18/04/2025", "13/06/2025", 350, 120.00),
                ("Final Destination: Bloodlines (Dolby Atmos Screening)", 3, "16/05/2025", "11/07/2025", 350, 120.00),
                ("A Minecraft Movie (3D Screening)", 4, "04/04/2025", "30/05/2025", 200, 100.00),
                ("Hurry Up Tomorrow (2D Screening)", 5, "16/05/2025", "11/07/2025", 200, 100.00),
                ("The Ugly Stepsister (2D Screening)", 6, "09/05/2025", "30/05/2025", 135, 100.00),
                ("Snow White - Live Action", 7, "21/03/2025", "23/05/2025", 80, 100.00)
            ]
            cursor.executemany('''
                INSERT INTO movies (title, cinema_room, release_date, end_date, tickets_available, ticket_price)
                VALUES (?, ?, ?, ?, ?, ?)
            ''', current_movies)
            conn.commit()
        conn.close()
    except Exception as e:   #error handling if a user were to enter invalid data into the field
        print("Could not write to database:", e)

def handle_client(conn, addr): #This class will be used for the server to handle client connections and sending the information in the JSON format as specified 
    print(f"[NEW CONNECTION] {addr} connected.")
    connected = True
    while connected:
        try:
            header = conn.recv(HEADER).decode(FORMAT)
            if not header:
                break

            msg_length = int(header.strip())
            msg = conn.recv(msg_length).decode(FORMAT)
            request = json.loads(msg)
            action = request.get("action")
            data = request.get("data")

            response = process_request(action, data)
            send_response(conn, response)

        except Exception as e:
            print(f"[ERROR] {addr} - {e}")
            break

    conn.close()


def process_request(action, data):  #prompts to be followed based on the if statement chosen
    try:
        conn = sqlite3.connect(DB_FILE)
        cursor = conn.cursor()

        if action == "Retrieve The List of Movies":
            cursor.execute("SELECT * FROM movies")
            rows = cursor.fetchall()
            return {"status": "Success", "Current Movies Showing": rows}

        elif action == "Add A New Movie":
            cursor.execute('''
                INSERT INTO movies (title, cinema_room, release_date, end_date, tickets_available, ticket_price)
                VALUES (?, ?, ?, ?, ?, ?)
            ''', (data["title"], data["cinema_room"], data["release_date"], data["end_date"], data["tickets_available"], data["ticket_price"]))
            conn.commit()
            return {"status": "Success", "message": "Movie added."}

        elif action == "Update Movie Details":
            cursor.execute('''
                UPDATE movies
                SET title = ?, cinema_room = ?, release_date = ?, end_date = ?, tickets_available = ?, ticket_price = ?
                WHERE id = ?
            ''', (data["title"], data["cinema_room"], data["release_date"], data["end_date"], data["tickets_available"], data["ticket_price"], data["id"]))
            conn.commit()
            return {"status": "Success", "message": "Movie updated."}

        elif action == "Delete A Movie":
            cursor.execute("DELETE FROM movies WHERE id = ?", (data["id"],))
            conn.commit()
            return {"status": "Success", "message": "Movie deleted."}

        elif action == "Record a Ticket Sale and Number of Tickets":
            movie_id = data["movie_id"]
            name = data["customer_name"]
            qty = data["number_of_tickets"]

            cursor.execute("SELECT ticket_price, tickets_available, title FROM movies WHERE id = ?", (movie_id,))
            row = cursor.fetchone()
            if not row:
                return {"status": "Error", "message": "Movie not found."}

            price, available, title = row #error handling output for the incidence of not enough tickets available 
            if qty > available:
                return {"status": "Error", "message": "Not enough tickets."}

            total = qty * price #price calculation of tickets based on the no. of tickets input to the system 
            cursor.execute('''
                INSERT INTO sales (movie_id, customer_name, number_of_tickets, total)
                VALUES (?, ?, ?, ?)
            ''', (movie_id, name, qty, total))
            cursor.execute("UPDATE movies SET tickets_available = tickets_available - ? WHERE id = ?", (qty, movie_id))
            conn.commit()

            filename = f"{name.replace(' ', '_')}_{movie_id}_ticket.txt"
            with open(filename, "w") as f:
                f.write(f"Movie: {title}\nCustomer: {name}\nTickets: {qty}\nTotal: R{total:.2f}")

            return {"Order Status": "Successful", "Total": total, "Movie Ticket": filename}

        else:
            return {"status": "Error", "message": "Unknown action."}
    except Exception as e:
        return {"status": "Error", "message": str(e)}
    finally:
        conn.close()

def send_response(conn, response):  #encoding the information recieved / sent into the JSON format as specified 
    res_json = json.dumps(response)
    res_len = len(res_json)
    send_header = str(res_len).encode(FORMAT)
    send_header += b' ' * (HEADER - len(send_header))
    conn.send(send_header)
    conn.send(res_json.encode(FORMAT))

def start():  #Threading will be used to allow for multiple client connection as dealing with a cinema, it is likely multiple people would want to make a booking simulataneously
    init_database()
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server.bind(ADDR)
    server.listen()
    print(f"[LISTENING] Server is listening on {SERVER}:{PORT}") #Server and Port will be taken from the gethost command at the start of the program
    while True:
        conn, addr = server.accept()
        thread = threading.Thread(target=handle_client, args=(conn, addr))
        thread.start()
        print(f"[ACTIVE CONNECTIONS] {threading.active_count() - 1}")

if __name__ == "__main__":
    print("[STARTING] server is starting...")
    start()


[STARTING] server is starting...
[LISTENING] Server is listening on 192.168.3.202:5050
[NEW CONNECTION] ('192.168.3.202', 62677) connected.
[ACTIVE CONNECTIONS] 9
